In [ ]:

# === PyTorch Core ===
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# === Optimizers & LR scheduling ===
import torch.optim as optim
from torch.optim import Adam, AdamW, SGD, Adagrad, RMSprop
from torch.optim.lr_scheduler import (
    StepLR, MultiStepLR, ExponentialLR, CosineAnnealingLR, ReduceLROnPlateau
)
# Keras의 regularizers.l2 -> PyTorch는 보통 optimizer의 weight_decay로 처리
# 예: AdamW(params, lr=1e-3, weight_decay=1e-4)


import torchvision
from torchvision import transforms
from torchvision import models  
from torchmetrics import (
    Accuracy, Precision, Recall, F1Score, JaccardIndex
)
from torchmetrics.segmentation import DiceScore as Dice


import os
import random
import numpy as np
from tqdm import tqdm
from skimage.io import imread, imshow
from skimage.morphology import label
from skimage.transform import resize
import matplotlib.pyplot as plt
import datetime
import sys
from contextlib import nullcontext
device = "cuda:0" if torch.cuda.is_available() else "cpu"

from data_load_BT import CovidDataset, datasets
from models1 import Encoder, Decoder, VAE
print(datasets)


In [ ]:
# BT pretrain — full BP split (lazy load)
batch_size = 64
data = datasets.bp_train_split(batch_size=batch_size, num_workers=2)

train_set, val_set, test_set = data.train, data.val, data.test
train_loader, val_loader, test_loader = data.train_loader, data.val_loader, data.test_loader

print(data)
print(len(train_set), len(val_set), len(test_set))


In [ ]:
import matplotlib.pyplot as plt
import torch

# DataLoader에서 한 배치 꺼내기
imgs, masks, lefts, rights = next(iter(train_loader))

# 첫 번째 배치의 첫 번째 샘플만 확인
i = 0
orig  = imgs[i]
left  = lefts[i]
right = rights[i]

# 텐서 → numpy
def to_np(x):
    return x.squeeze().detach().cpu().numpy()

plt.figure(figsize=(12,4))
for j, (im, title) in enumerate([(orig, "Main Aug"), (left, "Left Aug"), (right, "Right Aug")]):
    plt.subplot(1,3,j+1)
    plt.imshow(to_np(im), cmap='gray', vmin=0, vmax=1)
    plt.title(title)
    plt.axis('off')
plt.show()


In [ ]:

import torch
import math
import time
import math
import torch

# LOG2PI = math.log(2.0 * math.pi)

# def gaussian_logpdf(m, logvar, z):
#     """
#     m, logvar, z: (B, C, 1, 1) 또는 (B, C)
#     반환: (B, C)  각 채널별 log p(z|m, logvar)
#     """
#     # (B,C,1,1) -> (B,C)
#     if m.ndim == 4:
#         s = lambda t: t.squeeze(-1).squeeze(-1)
#         m, logvar, z = map(s, (m, logvar, z))

#     # 수치 안정화: logvar 클램프 & var 최소값 보호
#     logvar = logvar.clamp(-6.0, 1.0)
#     var = torch.exp(logvar)  # (B,C)

#     # log N(z; m, var) = -0.5 * [ (z-m)^2/var + log(2π) + log(var) ]
#     diff2 = (z - m) ** 2
#     logpdf = -0.5 * (diff2 / (var + 1e-12) + LOG2PI + logvar)
    
#     return logpdf  # (B, C)


# def barlow_from_logpdf(l1, l2, lambda_offdiag=0.005, eps=1e-4):
#     # l1, l2: (B, C) 또는 (B, C, 1, 1)
#     if l1.ndim == 4:
#         s = lambda t: t.squeeze(-1).squeeze(-1)
#         l1, l2 = map(s, (l1, l2))

#     B = l1.size(0)

#     # 배치 표준화(채널별 0-mean, 1-std)
#     z1 = (l1 - l1.mean(dim=0)) / (l1.std(dim=0) + eps)
#     z2 = (l2 - l2.mean(dim=0)) / (l2.std(dim=0) + eps)

#     # 교차상관행렬 C=(1/B) z1^T z2  → (C,C)
#     C = (z1.T @ z2) / float(B)

#     diag = torch.diag(C)
#     diag_loss = (1.0 - diag).pow(2).sum()           # 대각은 1로
#     off = C - torch.diag(diag)
#     off_loss = (off.pow(2)).sum()                   # 비대각은 0으로

#     return diag_loss + lambda_offdiag * off_loss




conv1x1 = nn.Conv2d(in_channels=2, out_channels=1, kernel_size=1)

# def concat_v1v2(view1, view2):
#     # (B,1,H,W) + (B,1,H,W) → (B,2,H,W)
#     view3 = torch.cat([view1, view2], dim=1)
#     # 1×1 conv로 채널을 2→1 축소
#     return conv1x1(view3)

# def elementwise_mul(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
#     return x * y   # torch.mul(x, y) 도 동일




# import math

# import torch

# def kl_diag_gauss(mu_p, logvar_p, mu_q, logvar_q, reduce='mean', eps=1e-12):
#     """
#     KL( N(mu_p, diag(exp(logvar_p))) || N(mu_q, diag(exp(logvar_q))) )
#     입력: (B,C,1,1) 또는 (B,C) 또는 (...,C)
#     반환: (B,) 또는 스칼라(mean/sum) 또는 전체( none )
#     """
#     # (...,C,1,1) -> (...,C)
#     if mu_p.ndim >= 3 and mu_p.shape[-1] == 1 and mu_p.shape[-2] == 1:
#         squeeze2 = lambda x: x.squeeze(-1).squeeze(-1)
#         mu_p, logvar_p, mu_q, logvar_q = map(squeeze2, (mu_p, logvar_p, mu_q, logvar_q))

#     var_p = torch.exp(logvar_p).clamp_min(eps)
#     var_q = torch.exp(logvar_q).clamp_min(eps)

#     diff2 = (mu_q - mu_p) ** 2
#     # per-dim KL
#     kl_per_dim = (var_p + diff2) / var_q + (logvar_q - logvar_p) - 1.0
#     kl = 0.5 * kl_per_dim.sum(dim=-1)  # sum over last dim (features)

#     if reduce == 'mean':
#         return kl.mean()
#     elif reduce == 'none':
#         return kl
#     elif reduce == 'sum':
#         return kl.sum()
#     else:
#         raise ValueError("reduce must be 'mean' | 'sum' | 'none'")

import torch

def kl_gaussian(mu1, sigma1, mu2, sigma2, eps: float = 1e-12):
    """
    KL( N(mu1, sigma1^2) || N(mu2, sigma2^2) )
    모든 텐서는 브로드캐스팅 가능. 반환값은 입력과 동일한 브로드캐스팅 shape.

    Args:
        mu1, sigma1, mu2, sigma2: 텐서 (예: [..., C])
        eps: 수치 안정성용 작은 값
    Returns:
        kl: 텐서 (mu/sigma와 동일한 브로드캐스팅 shape)
    """
    var1 = (sigma1 ** 2).clamp_min(eps)
    var2 = (sigma2 ** 2).clamp_min(eps)
    diff2 = (mu1 - mu2) ** 2
    return 0.5 * (torch.log(var2) - torch.log(var1) + (var1 + diff2) / var2 - 1.0)


def kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=3.0):
    """
    diag는 0으로, off는 target_off로. 각 항에 가중치 부여 가능.
    """
    C = K.size(0)
    eye = torch.eye(C, device=K.device, dtype=K.dtype).bool()
    diag_loss = (K.diagonal() ** 2).mean()                    # (K[ii]-0)^2
    off_loss  = ((K[~eye] - target_off) ** 2).mean()          # (K[ij]-1)^2, i!=j
    return w_diag * diag_loss + w_off * off_loss


def cross_corr_mu_loss(m1v, m2v, eps=1e-12, lambda_off=1.0):
    """
    μ 기반 cross-correlation loss
    Barlow Twins 구조(대각=1, 비대각=0)
    """
    # [B,C] 입력 가정
    B = m1v.size(0)
    # 배치 정규화
    z1n = (m1v - m1v.mean(0)) / (m1v.std(0) + eps)
    z2n = (m2v - m2v.mean(0)) / (m2v.std(0) + eps)

    # C matrix
    C = (z1n.T @ z2n) / float(B)  # [C,C]

    # 대각/비대각 분리
    D = C.size(0)

    diag = torch.diag(C)
    off  = C - torch.diag(diag)

    diag_loss = (1.0 - diag).pow(2).mean()                    # diag → 1
    off_loss  = off.pow(2).sum() / (D * (D - 1) + eps)         # off → 0

    return diag_loss + lambda_off * off_loss, C









In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from contextlib import nullcontext
import time


def pretrain_bt(
    encoder: nn.Module,
    train_loader,
    epochs: int = 200,
    lr: float = 1e-4,
    weight_decay: float = 1e-4,
    amp: bool = True,
    grad_clip: float | None = 1.0,
    ckpt_path: str | None = "bt_pretrain_bp.pth",
    print_every: int = 50,
    augment_on_gpu: bool = True,
    val_loader=None,   
):
    device = "cuda:2" if torch.cuda.is_available() else "cpu"
    encoder.to(device).train()
    conv1x1.to(device)

    # BN freeze
    for m in encoder.modules():
        if isinstance(m, torch.nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False

    if len(train_loader) == 0:
        raise RuntimeError("train_loader 길이가 0입니다. Dataset/DataLoader 구성을 확인하세요.")

    opt = torch.optim.AdamW(
        list(encoder.parameters()) + list(conv1x1.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )
    
    scaler = torch.cuda.amp.GradScaler(enabled=(amp and device.startswith("cuda")))

    def _progress_line(ep, epochs, it, total, loss, start_time, bar_len=30):
        ratio = it / max(total, 1)
        filled = max(0, min(int(ratio * bar_len), bar_len))
        bar = "█" * filled + "·" * (bar_len - filled)
        elapsed = time.time() - start_time
        ips = it / elapsed if elapsed > 0 else 0.0     # iters/sec
        eta = (total - it) / ips if ips > 0 else 0.0
        print(
            f"\r[ep {ep:3d}/{epochs:3d}] [{bar}] {it:4d}/{total:4d} "
            f"loss={loss:8.5f}  {elapsed:6.1f}s  ETA {eta:5.1f}s",
            end="", flush=True
        )
    best_val = float("inf")

    for ep in range(1, epochs + 1):
        running = 0.0
        epoch_start = time.time()
        encoder.train()
        for it, batch in enumerate(train_loader, 1):
            if isinstance(batch, (list, tuple)):
                view1 = batch[2]   
                view2 = batch[3]   
            else:
                raise ValueError("batch가 tuple이 아닙니다.")
            view1 = view1.to(device, non_blocking=True) #32 1 256 256 
            view2 = view2.to(device, non_blocking=True) #32 1 256 256 

           
            opt.zero_grad(set_to_none=True)
            use_amp = scaler.is_enabled()
            cm = torch.cuda.amp.autocast() if use_amp else nullcontext()

            with cm:
                if view1.ndim == 3: view1 = view1.unsqueeze(1)
                if view2.ndim == 3: view2 = view2.unsqueeze(1)
          

                assert view1.shape[1] == 1 and view2.shape[1] == 1, f"{view1.shape=}, {view2.shape=}"

                m1, v1_, z1 = encoder(view1) #각 32 1024 1 1 에서 -> 32 1024 
                m2, v2_, z2 = encoder(view2)

                v1_ = v1_.clamp(-5, 2); v2_ = v2_.clamp(-5, 2)

                
                # print("m1: ", m1.shape) (b,base^2,8,8)
                
                _gap = nn.AdaptiveAvgPool2d(1)
                s = lambda x: _gap(x).flatten(1) if x.dim() == 4 else x
                m1v, v1v, z1v = s(m1), s(v1_), s(z1)  # 32 1024 (base^2)
                m2v, v2v, z2v = s(m2), s(v2_), s(z2)  # 32 1024
              
                eps = 1e-12
                # m1v, v1v, m2v, v2v: [B, C]  (v*는 logvar)
                sig1 = (0.5 * v1v).exp()   # [B, C]
                sig2 = (0.5 * v2v).exp()   # [B, C]

                    # ➜ per-sample KL 행렬 만들고 배치 평균 내기
                K12_b = kl_gaussian(
                    mu1    = m1v.unsqueeze(2),   # [B,C,1]
                    sigma1 = sig1.unsqueeze(2),  # [B,C,1]
                    mu2    = m2v.unsqueeze(1),   # [B,1,C]
                    sigma2 = sig2.unsqueeze(1),  # [B,1,C]
                )                                # -> [B,C,C]

                K21_b = kl_gaussian(
                        mu1    = m2v.unsqueeze(2),
                        sigma1 = sig2.unsqueeze(2),
                        mu2    = m1v.unsqueeze(1),
                        sigma2 = sig1.unsqueeze(1),
                    )

                K = 0.5 * (K12_b + K21_b)        # [B,C,C]
                K = K.mean(dim=0)

                tgt = 1.0 + min(1.0, ep / 100.0)  # ep 0→100 동안 1.0→2.0

                loss_kl = kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=tgt)

                loss_cmu, C_mu = cross_corr_mu_loss(m1v, m2v, lambda_off=1.0)  #c_mu는 학습용 X,. 나중에 시각화용용
                lambda_cmu = 0.2  # 0.05 ~ 0.2 권장

                    # ③ 최종 Loss
                loss = loss_kl + lambda_cmu * loss_cmu

                
        
                                

                if not torch.isfinite(loss).item():
                    raise ValueError("Final loss is NaN/Inf")

            if use_amp:
                scaler.scale(loss).backward()
                if grad_clip is not None:
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(encoder.parameters(), grad_clip)
                scaler.step(opt)
                scaler.update()

            else:
                loss.backward()
                if grad_clip is not None:
                    nn.utils.clip_grad_norm_(encoder.parameters(), grad_clip)
                opt.step()

            # if it == 1:
            #     with torch.no_grad():
            #         tot = 0.0
            #         for p in encoder.parameters():
            #             if p.grad is not None:
            #                 tot += p.grad.detach().pow(2).sum().item()
                    

            running += loss.item()
            _progress_line(ep, epochs, it, len(train_loader), float(loss.item()), epoch_start)
        print()
        epoch_sec = time.time() - epoch_start
        avg = running / max(1, len(train_loader))
        print(f"=> epoch {ep}  time={epoch_sec:.2f}s  train_avg_loss={avg:.6f}")

        # ======== ★ 검증 루프 (val_loss 기준으로 저장) ========
        if val_loader is not None and len(val_loader) > 0:
            encoder.eval()
            val_running = 0.0
            with torch.no_grad():
                for vbatch in val_loader:
                    if isinstance(vbatch, (list, tuple)):
                        v1 = vbatch[2]
                        v2 = vbatch[3]
                    else:
                        raise ValueError("val batch가 tuple이 아닙니다.")
                    v1 = v1.to(device, non_blocking=True)
                    v2 = v2.to(device, non_blocking=True)
                   
                   

                    m1, v1_, z1 = encoder(v1)
                    m2, v2_, z2 = encoder(v2)
                    

                    v1_ = v1_.clamp(-5, 2); v2_ = v2_.clamp(-5, 2)
                    
                    _gap = nn.AdaptiveAvgPool2d(1)
                    s = lambda x: _gap(x).flatten(1) if x.dim() == 4 else x
                    m1v, v1v, z1v = s(m1), s(v1_), s(z1) # (B,C,1,1) → (B,C)
                    m2v, v2v, z2v = s(m2), s(v2_), s(z2) # (B,C,1,1) → (B,C)
                    eps = 1e-12
                    # m1v, v1v, m2v, v2v: [B, C]  (v*는 logvar)
                   

                    # 쌍방 KL → 대칭 KL 행렬 K: [C, C]
                    sig1 = (0.5 * v1v).exp()   # [B, C]
                    sig2 = (0.5 * v2v).exp()   # [B, C]

                    # ➜ per-sample KL 행렬 만들고 배치 평균 내기
                    K12_b = kl_gaussian(
                        mu1    = m1v.unsqueeze(2),   # [B,C,1]
                        sigma1 = sig1.unsqueeze(2),  # [B,C,1]
                        mu2    = m2v.unsqueeze(1),   # [B,1,C]
                        sigma2 = sig2.unsqueeze(1),  # [B,1,C]
                    )                                # -> [B,C,C]

                    K21_b = kl_gaussian(
                        mu1    = m2v.unsqueeze(2),
                        sigma1 = sig2.unsqueeze(2),
                        mu2    = m1v.unsqueeze(1),
                        sigma2 = sig1.unsqueeze(1),
                    )

                    K = 0.5 * (K12_b + K21_b)        # [B,C,C]
                    K = K.mean(dim=0)

                    tgt = 1.0 + min(1.0, ep / 100.0)  # ep 0→100 동안 1.0→2.0

                    loss_kl = kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=tgt)

                    loss_cmu, C_mu = cross_corr_mu_loss(m1v, m2v, lambda_off=1.0)  #c_mu는 학습용 X,. 나중에 시각화용용
                    lambda_cmu = 0.2  # 0.05 ~ 0.2 권장

                    # ③ 최종 Loss
                    loss = loss_kl + lambda_cmu * loss_cmu
                   
                    val_running += loss.item()
                    val_avg = val_running / max(1, len(val_loader))
                print(f"=> epoch {ep} val_loss={val_avg:.6f}")

            if ckpt_path and val_avg < best_val:
                best_val = val_avg
                torch.save({"ep": ep, "encoder": encoder.state_dict()}, ckpt_path)
                print(f"  ↳ saved BEST (by val_loss) to {ckpt_path} (val {best_val:.6f})")
       

In [ ]:
# === 실행 ===
device = "cuda:0" if torch.cuda.is_available() else "cpu"
encoder = Encoder(base=32).to(device).train()   # 이미 만들어둔 Encoder 인스턴스라면 이 줄만


pretrain_bt(
    encoder=encoder,
    train_loader=train_loader,
    val_loader=val_loader,
    
    epochs=200,            # 테스트로 1~3 epoch 먼저
    lr=3e-4,
    weight_decay=1e-4,
    amp=False,           # 처음엔 False 권장 (수치 이슈 확인 후 True)
    grad_clip=1.0,
    ckpt_path="bt_pretrain_bp.pth",
    print_every=1,       # 매 이터 로그
    augment_on_gpu=True # get_two_views가 CPU 텐서 기준이면 False
)

In [ ]:
# # ===== C(교차상관) 계산 & 시각화 유틸 =====
# import torch
# import torch.nn.functional as F
# import matplotlib.pyplot as plt
# import numpy as np

# @torch.no_grad()
# def compute_C_mu(encoder, batch):
#     """
#     batch에서 view1, view2를 꺼내 encoder 통과 → μ만으로 C 계산.
#     (네 데이터 구조가 batch[2], batch[3]에 두 view라고 가정)
#     반환: C [C,C]
#     """
#     dev = next(encoder.parameters()).device
#     encoder.eval()

#     v1, v2 = batch[2], batch[3]
#     if v1.ndim == 3: v1 = v1.unsqueeze(1)
#     if v2.ndim == 3: v2 = v2.unsqueeze(1)
#     v1, v2 = v1.to(dev, non_blocking=True), v2.to(dev, non_blocking=True)

#     m1, logv1, z1 = encoder(v1)  # [B,C,H,W] (H=W=8 같은 값)
#     m2, logv2, z2 = encoder(v2)

#     # 네가 쓰는 GAP 경로 그대로
#     s = lambda x: F.adaptive_avg_pool2d(x, 1).flatten(1) if x.dim()==4 else x
#     x1, x2 = s(m1), s(m2)       # [B,C]

#     eps = 1e-4
#     x1n = (x1 - x1.mean(0)) / (x1.std(0) + eps)
#     x2n = (x2 - x2.mean(0)) / (x2.std(0) + eps)
#     B   = x1n.size(0)
#     C   = (x1n.transpose(0,1) @ x2n) / float(B)  # [C,C]
#     return C

# def show_C_heatmap(C, title="Cross-Correlation C (mu)"):
#     C_np = C.detach().float().cpu().numpy()
#     plt.figure(figsize=(6,5))
#     im = plt.imshow(C_np, vmin=-1.0, vmax=1.0, origin="upper", interpolation="nearest")
#     plt.colorbar(im, fraction=0.046, pad=0.04)
#     plt.title(title)
#     plt.tight_layout()
#     plt.show()

#     # 통계 출력(대각 ~1, 비대각 ~0 확인용)
#     D = C_np.shape[0]
#     diag = np.diag(C_np)
#     off  = C_np - np.diag(diag)
#     off_vals = off[np.triu_indices(D, k=1)]
#     print(
#         f"[C stats] diag: mean={diag.mean():.4f}, std={diag.std():.4f} | "
#         f"off: mean={off_vals.mean():.4f}, rms={np.sqrt((off_vals**2).mean()):.4f}, "
#         f"max|off|={np.abs(off_vals).max():.4f}"
#     )


# batch = next(iter(train_loader))
# C = compute_C_mu(encoder, batch)
# show_C_heatmap(C, title="Cross-Correlation C (mu)")


# # ====== viz_utils.py에 추가(또는 별도 셀) ======
# import torch
# import torch.nn as nn
# import numpy as np
# import matplotlib.pyplot as plt

# # (이미 있다면 중복 정의 불필요)
# def _model_device(m: torch.nn.Module):
#     for p in m.parameters(): return p.device
#     for b in m.buffers():    return b.device
#     return torch.device("cpu")

# @torch.no_grad()
# def _prep_views_from_batch(batch, device):
#     v1, v2 = batch[2], batch[3]   # 너의 데이터 포맷 전제
#     if v1.ndim == 3: v1 = v1.unsqueeze(1)
#     if v2.ndim == 3: v2 = v2.unsqueeze(1)
#     return v1.to(device, non_blocking=True), v2.to(device, non_blocking=True)

# # --- KL 대칭행렬 계산 ---
# @torch.no_grad()
# def compute_K_once(encoder, batch, symmetric=True, reduce="mean"):
#     """
#     반환: K [C,C] (reduce='mean' 또는 'sum'), 혹은 [B,C,C] (reduce='none')
#     """
#     dev = _model_device(encoder)
#     encoder.eval()
#     v1, v2 = _prep_views_from_batch(batch, dev)

#     m1, logv1, z1 = encoder(v1)     # [B,C,H,W]
#     m2, logv2, z2 = encoder(v2)

#     _gap = nn.AdaptiveAvgPool2d(1)
#     s = lambda x: _gap(x).flatten(1) if x.dim()==4 else x

#     m1v, m2v = s(m1), s(m2)         # [B,C]
#     lv1, lv2 = s(logv1), s(logv2)   # [B,C]
#     sig1 = (0.5 * lv1).exp()
#     sig2 = (0.5 * lv2).exp()

#     def kl_gaussian(mu1, sigma1, mu2, sigma2, eps=1e-12):
#         v1 = (sigma1**2).clamp_min(eps)
#         v2 = (sigma2**2).clamp_min(eps)
#         d2 = (mu1 - mu2) ** 2
#         return 0.5 * (torch.log(v2) - torch.log(v1) + (v1 + d2) / v2 - 1.0)

#     # [B,C,C]
#     K12_b = kl_gaussian(m1v.unsqueeze(2), sig1.unsqueeze(2),
#                         m2v.unsqueeze(1), sig2.unsqueeze(1))
#     if symmetric:
#         K21_b = kl_gaussian(m2v.unsqueeze(2), sig2.unsqueeze(2),
#                             m1v.unsqueeze(1), sig1.unsqueeze(1))
#         K_b = 0.5 * (K12_b + K21_b)
#     else:
#         K_b = K12_b

#     if reduce == "mean":
#         return K_b.mean(dim=0)      # [C,C]
#     elif reduce == "sum":
#         return K_b.sum(dim=0)       # [C,C]
#     else:
#         return K_b                   # [B,C,C]

# # --- 히트맵 표시 ---
# def show_K_heatmap(K, title="Symmetric KL (mean)", vmax_pct=99, vmin=0.0):
#     K_np = K.detach().float().cpu().numpy() if hasattr(K, "detach") else np.asarray(K)
#     vmax = float(np.percentile(K_np, vmax_pct))
#     plt.figure(figsize=(6,5))
#     im = plt.imshow(K_np, vmin=vmin, vmax=vmax, origin="upper", interpolation="nearest")
#     plt.colorbar(im, fraction=0.046, pad=0.04)
#     plt.title(f"{title} | vmax@p{vmax_pct}={vmax:.3f}")
#     plt.tight_layout()
#     plt.show()

#     D = K_np.shape[0]
#     diag = np.diag(K_np)
#     off  = K_np[~np.eye(D, dtype=bool)]
#     print(f"[K stats] diag mean={diag.mean():.4f}, std={diag.std():.4f} | "
#           f"off mean={off.mean():.4f}, p90={np.percentile(off,90):.4f}, "
#           f"p99={np.percentile(off,99):.4f}, max={off.max():.4f}")
    
# batch = next(iter(train_loader))
# K = compute_K_once(encoder, batch, symmetric=True, reduce="mean")
# show_K_heatmap(K, title="Symmetric KL (mean)")




In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as D
import torch.optim as optim
from torch import Tensor
import os
# For displaying and evaluating results.
import numpy as np
from matplotlib import pyplot as plt

# For monitoring resource-usage and progress.
from timeit import default_timer as timer
import psutil
from os.path import join

# GPU 번호 지정 (0번 GPU 사용)

# ===== Paths: 너의 데이터 구조에 맞게 수정 =====
# 데이터셋 루트 (train/val/test 및 각 original/label 이 안에 존재)
root_dir = 'datasets/bp_data/train_30_split'

# 기존 코드에서 raw_data_dir이 '원본 데이터 상위 폴더' 개념이라면, 
# 지금은 이미 train/val/test 구조가 준비되어 있으니 루트 자체를 가리키게 함
raw_data_dir = root_dir

# 전처리/로그/모델 저장 디렉토리들은 같은 루트 밑에 만들어 두자
prep_data_dir    = join(root_dir, 'prep_data')
losses_dir       = join(root_dir, 'losses')
models_dir       = join(root_dir, 'saved_models')
checkpoint_dir   = join(root_dir, 'checkpoints')

# (선택) DataLoader가 명시적인 이미지/마스크 경로를 요구한다면 아래도 활용 가능
train_img_dir = join(root_dir, 'train', 'original')
train_mask_dir = join(root_dir, 'train', 'label')
val_img_dir   = join(root_dir, 'val', 'original')
val_mask_dir  = join(root_dir, 'val', 'label')
test_img_dir  = join(root_dir, 'test', 'original')
test_mask_dir = join(root_dir, 'test', 'label')

# 디렉토리 보장
os.makedirs(prep_data_dir,  exist_ok=True)
os.makedirs(losses_dir,     exist_ok=True)
os.makedirs(models_dir,     exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)


In [ ]:
# VAE fine-tune — BP train 30% (lazy load)
# COVID로 바꾸려면: datasets.covid_90p(...) 또는 datasets.covid(...)
batch_size = 16
data = datasets.bp_train_30(batch_size=batch_size, num_workers=2)
# data = datasets.covid_90p(batch_size=batch_size, num_workers=2)

train_set, val_set, test_set = data.train, data.val, data.test
train_loader, val_loader, test_loader = data.train_loader, data.val_loader, data.test_loader

print(data)
print(len(train_set), len(val_set), len(test_set))


In [ ]:
def make_seg_loaders(data, batch_size):
    N = len(data); N_train = int(0.9*N); 
    N_dev = int((N - N_train)); 
    print(N_train)
    print(N_dev)
    train_data, dev_data = D.random_split(
        data, [N_train, N_dev])

    train_loader = D.DataLoader(
        train_data, batch_size=batch_size, shuffle=True)
 
    dev_loader = D.DataLoader(
        dev_data, batch_size=batch_size, shuffle=True)
    print(len(train_loader))
    print(len(dev_loader))      
    return train_loader, dev_loader

In [ ]:
import torch
import torch.optim as optim
from itertools import chain
import torch.nn as nn  # ← 추가

# 1) VAE 모델 초기화
vae = VAE(base=32).to(device)  # 그대로 유지

# 2) Encoder 가중치 로드
ckpt_path = "bt_pretrain_bp.pth"  # Cell 5에서 저장한 파일
ckpt = torch.load(ckpt_path, map_location=device)

enc_state = ckpt["encoder"] if isinstance(ckpt, dict) and "encoder" in ckpt else ckpt

def _strip_module_prefix(state):
    return { (k[len("module."):] if k.startswith("module.") else k): v
             for k, v in state.items() }

enc_state = _strip_module_prefix(enc_state)

# 로드는 기존 encoder 모듈에 먼저 진행
missing, unexpected = vae.encoder.load_state_dict(enc_state, strict=False)
print(f"[encoder load] missing={len(missing)} unexpected={len(unexpected)}")

# === (수정) 입력 어댑터: in_ch → exp_in 을 '필요할 때만' prepend ===
def _first_conv_in_channels(m: nn.Module) -> int:
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            return mod.in_channels
    raise RuntimeError("No Conv2d found in encoder")

# 인코더가 기대하는 입력 채널 (보통 1)
exp_in = _first_conv_in_channels(vae.encoder)

# 실제 데이터의 입력 채널 추정 (train_loader에서 한 배치)
try:
    _sample_batch = next(iter(train_loader))
    _flair = _sample_batch[0]  # (flair, seg, left, right) 가정
    if _flair.ndim == 3:          # [B,H,W]
        in_ch = 1
    elif _flair.ndim == 4:        # [B,C,H,W]
        in_ch = _flair.shape[1]
    else:
        raise ValueError(f"Unexpected flair shape: {tuple(_flair.shape)}")
except Exception as e:
    print(f"[warn] fail to infer in_ch from train_loader: {e}")
    in_ch = exp_in  # 추정 실패 시 어댑터 생략

# 필요할 때만 어댑터 추가
if in_ch != exp_in:
    input_adapter = nn.Conv2d(in_channels=in_ch, out_channels=exp_in, kernel_size=1, bias=False).to(device)
    nn.init.kaiming_normal_(input_adapter.weight, nonlinearity='relu')
    vae.encoder = nn.Sequential(input_adapter, vae.encoder).to(device)
    print(f"[adapter] prepend 1x1 Conv: {in_ch} -> {exp_in}")
else:
    print(f"[adapter] skipped (in_ch == exp_in == {exp_in})")

# 3) Optimizer / Scheduler (변경 없음, 단 어댑터 파라미터가 포함되도록 '붙인 후' 생성)
optimizers = torch.optim.AdamW(vae.parameters(), lr=1e-4, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizers,
    mode='min',
    factor=0.95,
    patience=3,
    verbose=True,
    min_lr=1e-8
)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def binary_dice_loss_sigmoid(pred, target, smooth=1e-6):
    """
    Binary Dice Loss 계산 (시그모이드 기반)
    pred: [B, 1, H, W] - 로짓 값 (Sigmoid 미적용)
    target: [B, 1, H, W] - 단일 채널 마스크 (0: 배경, 1: 전경)
    """
    # Sigmoid 적용
    pred = torch.sigmoid(pred)  # [B, 1, H, W]
    
    # Dice 계산 (채널 차원을 유지하면서 계산)
    pred_flat = pred.reshape(pred.size(0), -1)  # [B, H*W]
    target_flat = target.reshape(target.size(0), -1)  # [B, H*W]

    intersection = (pred_flat * target_flat).sum(dim=1)  # [B]
    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)  # [B]
    
    dice = (2. * intersection + smooth) / (union + smooth)  # [B]
    loss = 1 - dice.mean()  # scalar
    
    return loss

def combined_binary_loss_sigmoid(pred, target, dice_weight=0.5, ce_weight=0.5):
    """
    이진 분류를 위한 결합 손실 (시그모이드 기반)
    pred: [B, 1, H, W] - logits
    target: [B, 1, H, W] - 단일 채널 마스크 (0: 배경, 1: 전경)
    """
   
    # Dice Loss 계산
    dice = binary_dice_loss_sigmoid(pred, target)
    
    # BCE Loss 계산 (CrossEntropy 대신)
    bce_loss = nn.BCEWithLogitsLoss()
    
    # pred와 target을 같은 형태로 reshape
    pred_reshaped = pred.view(pred.size(0), -1)  # [B, H*W]
    target_reshaped = target.view(target.size(0), -1)  # [B, H*W]
    ce = bce_loss(pred_reshaped, target_reshaped)

    # 전체 loss 계산
    total_loss = dice_weight * dice + ce_weight * ce
    
    return total_loss, dice.item(), ce.item()

def train_epoch(seg_model, optimizer, train_loader, epoch, device):
    seg_model.train()
    total_loss = 0
    total_recon = 0
    total_kl_loc = 0
    
    beta_local = 0.01
    recon_weight = 0.99

    for batch in train_loader:
        flair, seg ,_,_= batch
        flair = flair.to(device)
        seg = seg.to(device)
        flair = flair.unsqueeze(1)
        seg = seg.unsqueeze(1)
        
        
        # gradient 초기화
        optimizer.zero_grad()
        
        # Forward pass
        output, mu_local, logvar_local = seg_model(flair)

        # Reconstruction loss (dice loss) - 튜플 반환값 처리
        dice_loss_combined, dice_value, ce_value = combined_binary_loss_sigmoid(output, seg)
        
        # KL loss
        kl_loss_local = 1 + logvar_local - torch.square(mu_local) - torch.exp(logvar_local)
        kl_loss_local = torch.sum(kl_loss_local, dim=-1)
        kl_loss_local *= -0.5

        # Calculate means
        loss_k_loc = torch.mean(kl_loss_local)
        loss_d = dice_loss_combined  # 이미 평균 계산된 값

        # Total loss - 각 배치별로 계산
        loss = recon_weight * dice_loss_combined + beta_local * torch.mean(kl_loss_local)

        # Accumulate losses
        total_kl_loc += loss_k_loc.item()
        total_recon += loss_d.item()
        total_loss += loss.item()

        # Backward pass
        loss.backward()
        optimizer.step()

    print('train_total_local_kl: ', total_kl_loc/len(train_loader))
    print('train_total_dice: ', total_recon/len(train_loader))
    
    return total_loss / len(train_loader)

def evaluate(seg_model, dev_loader, epoch, save_dir, scheduler, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    seg_model.eval()
    total_loss = 0
    total_recon = 0
    total_kl_loc = 0
    beta_local = 0.01
    recon_weight = 0.99
    
    with torch.no_grad():
        for batch in dev_loader:
            flair, y, _, _ = batch
            flair = flair.to(device)
            flair = flair.unsqueeze(1)
            y = y.to(device)
            y = y.unsqueeze(1)
            
            # Evaluation
            output, mu_local, logvar_local = seg_model(flair)
     
            # 평가 시에도 binary loss 사용 (일관성 유지)
            dice_loss_combined, dice_value, ce_value = combined_binary_loss_sigmoid(output, y)
            
            kl_loss_local = 1 + logvar_local - torch.square(mu_local) - torch.exp(logvar_local)
            kl_loss_local = torch.sum(kl_loss_local, dim=-1)
            kl_loss_local *= -0.5
        
            loss_k_loc = torch.mean(kl_loss_local)
            loss_d = dice_loss_combined
            
            loss = recon_weight * dice_loss_combined + beta_local * torch.mean(kl_loss_local)

            total_kl_loc += loss_k_loc.item()
            total_recon += loss_d.item()
            total_loss += loss.item()

    avg_loss = total_loss / len(dev_loader)
    print('dev_total_local_kl: ', total_kl_loc/len(dev_loader))
    print('dev_total_dice: ', total_recon/len(dev_loader))

    scheduler.step(avg_loss)
    return avg_loss





In [ ]:
printed = False

for bidx, (flair, seg, left, right) in enumerate(train_loader):  # ← 반환되는 4개 다 받음
    flair = flair.to(device)
    
    seg   = seg.to(device)
    # left, right는 필요 없다면 그냥 무시하거나 아래처럼:
    # _, _, left, right = batch  로도 가능
    flair = flair.unsqueeze(1)

    print(flair.shape)

    # 1) 단계별로 쪼개서 실행
    x = flair  # [B,1,256,256]

    # initial
    x = vae.encoder.encoder_initial_flair(x)
    print("[after initial]", tuple(x.shape))  # 기대: (B, 32, 256, 256)  ← base=32 가정

    # features를 구성요소별로 추출
    features = vae.encoder.encoder_features_flair
    for i, m in enumerate(features):
        x = m(x)
        print(f"[after features[{i}] {m.__class__.__name__}]", tuple(x.shape))

    out, mu, logvar = vae(flair)

    if not printed:
        print(f"[DEBUG] out:    {tuple(out.shape)}")
        print(f"[DEBUG] mu:     {tuple(mu.shape)}")
        print(f"[DEBUG] logvar: {tuple(logvar.shape)}")
        printed = True


In [ ]:
# 학습 설정
total_epochs = 400
train_losses = []
dev_losses = []
best_loss = float('inf')

# 결과 저장 디렉토리 설정
results_dir = 'results/result_BP'
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

try:
    for epoch in range(1, total_epochs + 1):
        start_time = timer()
        
        # 학습 단계
        train_loss = train_epoch(vae, optimizers, train_loader, epoch, device)
        train_losses.append(train_loss)
        
        # 평가 단계
        dev_loss = evaluate(vae, val_loader, epoch, results_dir, scheduler, device)
        dev_losses.append(dev_loss)
        
        scheduler.step(dev_loss)
        
        # 최고 성능 모델 저장
        if dev_loss < best_loss:
            best_loss = dev_loss
            torch.save(
                vae.state_dict(),
                join(results_dir, 'VAE_model_best_BP_30.pt')
            )
            print(f"새로운 최고 성능 모델 저장됨. Loss: {best_loss:.4f}")
        
        # 에폭 통계 출력
        end_time = timer()
        print(
            f"에폭 {epoch}: "
            f"학습 손실: {train_loss:.4f}, "
            f"검증 손실: {dev_loss:.4f}, "
            f"에폭 소요 시간 = {(end_time - start_time):.3f}초"
        )
    
    print('학습이 완료되었습니다.')

except KeyboardInterrupt:
    print('사용자에 의해 학습이 중단되었습니다.')